# 07 · Evaluación en fotografía histórica

**Qué hace:** evaluación ciega (sin referencia) del pipeline completo sobre 21 fotografías históricas reales — genera máscaras de daño, restaura con LaMa + A-ESRGAN, mide NIQE / BRISQUE / Ma / PI / MUSIQ por imagen y condición, aplica contrastes de Wilcoxon y comprueba el solapamiento de las máscaras con rostros.

**Qué necesita:**
- Fotografías históricas — Release `v1.0` `vintage_degraded.zip` (54 imágenes; el notebook filtra a 21 según `artifacts/historicas_21.txt`)
- Checkpoints — Release `v1.0`: `lama_finetuned.zip`, `aesrgan_finetuned.zip`
- GPU (Google Colab)

**Qué deja escrito:** máscaras, métricas (CSV/JSON) y figuras en `ROOT/_out/07/{mascaras,metricas,figuras}/`. El bucle de máscaras es reanudable: si una máscara ya existe, esa imagen se salta.

**Arranque:** primera celda de código.

## 1. Entorno, dependencias y rutas


In [ ]:
!pip -q install simple-lama-inpainting "basicsr>=1.3.3.11" pyiqa lpips opencv-python scikit-image pandas scipy
from colab_setup import aplicar_parches, preparar_repo, descargar_datos, montar_drive_opcional
aplicar_parches()
ROOT = preparar_repo()
DATA = descargar_datos("release:vintage_degraded", "release:lama_ft", "release:aesrgan_ft", root=ROOT)
SALIDA = ROOT / "_out" / "07"
for sub in ("mascaras", "metricas", "figuras"):
    (SALIDA / sub).mkdir(parents=True, exist_ok=True)

In [ ]:
import base64, gc, io, json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageFile
from scipy.stats import wilcoxon

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ── Paleta Okabe-Ito (misma que en Fase 4d y 5.4, para coherencia de figuras) ────────────────
OI_BLUE   = '#0072B2'
OI_ORANGE = '#E69F00'
OI_GREEN  = '#009E73'
OI_PINK   = '#CC79A7'
OI_GREY   = '#999999'

matplotlib.rcParams.update({
    'font.family':     'sans-serif',
    'font.size':       9,
    'axes.labelsize':  9,
    'axes.titlesize':  9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi':      150,
    'savefig.dpi':     300,
    'savefig.bbox':    'tight',
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def torch_load(ruta, map_location='cpu'):
    """torch.load compatible con torch >= 2.6, donde weights_only pasó a True por defecto.

    Los checkpoints de BasicSR y el del ajuste fino de LaMa son diccionarios pickled
    con objetos que el cargador restringido rechaza.
    """
    try:
        return torch.load(ruta, map_location=map_location, weights_only=False)
    except TypeError:                      # torch < 1.13 no conoce el argumento
        return torch.load(ruta, map_location=map_location)


print('torch:', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')


In [ ]:
# ── Rutas ────────────────────────────────────────────────
import subprocess

# Carpeta real: mezcla degradación histórica y moderna (ver introducción). El filtrado por
# terminación de nombre de fichero ocurre en la sección 2, antes de anotar o procesar nada.
# El Release vintage_degraded.zip trae las 54 fotografías; el zip puede anidarlas en un
# subdirectorio, así que se resuelve al directorio que las contiene.
_EXTS_REAL = ('*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp')
_imgs_reales = sorted(p for ext in _EXTS_REAL
                      for p in DATA["release:vintage_degraded"].rglob(ext))
assert _imgs_reales, f'No hay imágenes en {DATA["release:vintage_degraded"]}'
DIR_REAL = _imgs_reales[0].parent

# ── Checkpoints ──────────────────────────────────────────────
# Ajustados: del Release v1.0 del TFM (los mismos que en el notebook 06 / apartado 5.4).
PTH_LAMA_FT     = next(DATA["release:lama_ft"].rglob("*.pth"))
PTH_AESRGAN_FT  = sorted(DATA["release:aesrgan_ft"].rglob("*.pth"))[-1]

# A-ESRGAN preentrenado: pesos oficiales del repo original (no están en el Release del TFM).
PTH_AESRGAN_PRE = SALIDA / 'A_ESRGAN_Single.pth'
if not PTH_AESRGAN_PRE.exists():
    subprocess.run(
        ["wget", "-q", "-O", str(PTH_AESRGAN_PRE),
         "https://github.com/stroking-fishes-ml-corp/A-ESRGAN/releases/download/v1.0.0/A_ESRGAN_Single.pth"],
        check=True,
    )

# Salidas del apartado 6.1/6.2.
DIR_ENTRADA  = SALIDA / 'entrada'        # fotografía real, solo redimensionada
DIR_MASCARA  = SALIDA / 'mascaras'       # máscara binaria pintada a mano (sección 3)
DIR_LAMA_PRE = SALIDA / 'lama_pre'
DIR_LAMA_FT  = SALIDA / 'lama_ft'
DIR_SR_PRE   = SALIDA / 'sr_pre'
DIR_SR_FT    = SALIDA / 'sr_ft'
DIR_METRICAS = SALIDA / 'metricas'
DIR_FIGURAS  = SALIDA / 'figuras'

for d in (DIR_ENTRADA, DIR_MASCARA, DIR_LAMA_PRE, DIR_LAMA_FT,
          DIR_SR_PRE, DIR_SR_FT, DIR_METRICAS, DIR_FIGURAS):
    d.mkdir(parents=True, exist_ok=True)

# Límite de lado mayor de entrada, igual que en el demostrador (apartado 5.5): con escala ×4,
# 1024 px de entrada ya produce 4096 px de salida.
RESIZE_MAX = 1024

print('Comprobación de rutas de entrada:')
for p, etiq in [(DIR_REAL, 'carpeta de fotografías (histórica + moderna, sin filtrar)'),
                (PTH_AESRGAN_PRE, 'A-ESRGAN preentrenado'),
                (PTH_AESRGAN_FT, 'A-ESRGAN F_flick iter 400'),
                (PTH_LAMA_FT, 'LaMa ajustado')]:
    print(f'  {"✓" if p.exists() else "✗ FALTA"}  {etiq}: {p}')

print(f'\nSalidas en: {SALIDA}')

## 2. Conjunto de evaluación

`Color_Phase0/sonda/real` no existía con ese contenido; el conjunto real vive en
`datasets/Vintage_Degraded` y mezcla **54** fotografías: unas históricas reales degradadas por el
tiempo, otras fotografías modernas degradadas artificialmente para otro propósito, ambas con el
mismo patrón de nombre (`low-resolution-photographs_imgN`). No hay forma automática de
distinguirlas por metadatos, así que el filtro es una lista de terminaciones de nombre de fichero
identificadas a mano.

**Este notebook filtra en el origen**, antes de anotar máscaras o correr el pipeline: así no se
anota ni se procesa ninguna de las 33 fotografías modernas, y no hay más adelante ningún riesgo de
que una tabla o una figura mezcle sin querer las 54 con las 21. `df_conjunto` nace ya con 21 filas.

Antes de seguir:

1. Comprobar que las 21 terminaciones de `TERMINACIONES_HISTORICAS` casan con 21 ficheros
   distintos (la celda siguiente avisa si no).
2. **Comprobar la separación frente al corpus de calibración espectral de 5.3.** Si alguna de
   estas 21 fotografías se usó para calibrar el módulo de degradación (`psd_objetivo.npz`), esa
   imagen no es una evaluación independiente. Rellenar `NOMBRES_CALIBRACION` con los nombres de
   fichero (sin extensión) del corpus de calibración de 5.3 antes de seguir.


In [ ]:
EXTS = ('*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp')

def listar(directorio):
    return sorted(p for ext in EXTS for p in Path(directorio).glob(ext))

# Terminaciones de nombre de fichero (sin extensión) de las fotografías verdaderamente históricas
# dentro de DIR_REAL. Identificadas a mano por Javier a partir de inspección visual del conjunto
# completo (54 imágenes, mezcla de degradación antigua real y moderna artificial); vendorizadas
# en artifacts/historicas_21.txt (una por línea) para no arrastrar la lista embebida en el notebook.
TERMINACIONES_HISTORICAS = [
    l.strip()
    for l in (ROOT / "artifacts" / "historicas_21.txt").read_text().splitlines()
    if l.strip()
]

def _es_historica(stem, terminaciones=TERMINACIONES_HISTORICAS):
    return any(stem.endswith(t) for t in terminaciones)

# Nombres de fichero (sin extensión) del corpus de calibración espectral de 5.3, si se conocen.
NOMBRES_CALIBRACION = set()

rutas_todas = listar(DIR_REAL)
assert rutas_todas, f'No hay imágenes en {DIR_REAL}'
print(f'{len(rutas_todas)} imágenes encontradas en {DIR_REAL} (histórica + moderna, sin filtrar).')

rutas_reales = [p for p in rutas_todas if _es_historica(p.stem)]
descartadas = [p.stem for p in rutas_todas if not _es_historica(p.stem)]
print(f'{len(rutas_reales)} identificadas como verdaderamente históricas por TERMINACIONES_HISTORICAS.')
print(f'{len(descartadas)} descartadas por degradación moderna (no se anotan ni se procesan).')

no_encontradas = sorted(t for t in TERMINACIONES_HISTORICAS
                        if not any(p.stem.endswith(t) for p in rutas_reales))
if no_encontradas:
    print(f'[AVISO] {len(no_encontradas)} terminaciones de la lista no aparecen en la carpeta: '
          f'{no_encontradas} — revisa el nombre exacto del fichero.')
if len(rutas_reales) != len(TERMINACIONES_HISTORICAS):
    print(f'[AVISO] Se esperaban {len(TERMINACIONES_HISTORICAS)} fotografías históricas; se '
          f'encontraron {len(rutas_reales)}. Puede haber una terminación que casa con más de un '
          'fichero — revisar antes de continuar.')

solapan = sorted(p.stem for p in rutas_reales if p.stem in NOMBRES_CALIBRACION)
if solapan:
    print(f'[AVISO CRÍTICO] {len(solapan)} imágenes coinciden con NOMBRES_CALIBRACION: {solapan}')
    print('  Sácalas del conjunto de evaluación o confirma que NOMBRES_CALIBRACION está mal.')
elif not NOMBRES_CALIBRACION:
    print('[AVISO] NOMBRES_CALIBRACION está vacío: la separación frente al corpus de calibración '
          'de 5.3 no se ha podido comprobar automáticamente. Confirmarlo a mano antes del depósito.')
else:
    print('Comprobación de separación frente al corpus de calibración: sin solapamiento.')

# ── Preprocesado: redimensionar si el lado mayor supera RESIZE_MAX (criterio del demostrador) ──
registros_conjunto = []
for ruta in rutas_reales:
    stem = ruta.stem
    img = Image.open(ruta).convert('RGB')
    w0, h0 = img.size
    escala = min(1.0, RESIZE_MAX / max(w0, h0))
    if escala < 1.0:
        img = img.resize((max(1, round(w0 * escala)), max(1, round(h0 * escala))), Image.LANCZOS)
    img.save(DIR_ENTRADA / f'{stem}.png')
    registros_conjunto.append({
        'imagen': stem, 'ancho_original': w0, 'alto_original': h0,
        'ancho': img.size[0], 'alto': img.size[1], 'redimensionada': escala < 1.0,
    })

df_conjunto = pd.DataFrame(registros_conjunto).set_index('imagen')
n_redim = int(df_conjunto['redimensionada'].sum())
print(f'\n{len(df_conjunto)} imágenes históricas preparadas en {DIR_ENTRADA} '
      f'({n_redim} redimensionadas a un máximo de {RESIZE_MAX} px de lado mayor).')
df_conjunto.to_csv(DIR_METRICAS / 'conjunto_6.csv')


In [ ]:
# Inspección visual del conjunto de entrada (4 ejemplos).
muestras = df_conjunto.index[:4]
fig, axes = plt.subplots(1, len(muestras), figsize=(3 * len(muestras), 3.2))
if len(muestras) == 1:
    axes = [axes]
for ax, stem in zip(axes, muestras):
    img = cv2.cvtColor(cv2.imread(str(DIR_ENTRADA / f'{stem}.png')), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f'{stem[:16]}\n{df_conjunto.loc[stem, "ancho"]}×{df_conjunto.loc[stem, "alto"]}',
                 fontsize=7)
    ax.axis('off')
plt.suptitle('Muestra del conjunto de evaluación — fotografía histórica real (sin procesar)',
             fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Anotación manual de las máscaras de daño

Estas 21 fotografías no traen máscara: no hay ningún generador sintético detrás que la produzca,
así que hay que marcar a mano dónde está el daño (roturas, arañazos, manchas, pérdidas de
emulsión), igual que haría un usuario del demostrador (5.5), pero aquí sobre las 21 imágenes del
conjunto de evaluación en lugar de una sola.

**Cómo funciona.** La celda siguiente define una herramienta de pintado embebida en el propio
notebook, en HTML/JavaScript sobre el lienzo del navegador (funciona en Colab; no funciona en un
entorno Jupyter local sin salida de navegador). Para cada fotografía:

1. Se muestra la imagen a un tamaño manejable en pantalla.
2. Se pinta con el ratón (o el dedo, en pantalla táctil) sobre las zonas dañadas; el grosor del
   pincel es ajustable con el deslizador.
3. «Deshacer» retira el último trazo; «Borrar todo» reinicia la máscara de esa imagen.
4. «Guardar máscara» cierra esa imagen y pasa a la siguiente.

El bucle es **reanudable**: si una máscara ya existe en `Fase6/mascaras/`, esa imagen se salta sin
volver a pedir anotación. Para repetir la anotación de una imagen concreta, borrar su fichero de
`Fase6/mascaras/` antes de reejecutar.

**Algunas fotografías no tienen daño localizado que pintar** (degradación global — desenfoque,
pérdida de detalle — sin rotura ni mancha) y es correcto dejarlas con máscara vacía; la sección 9.2
excluye esos casos de la comparación cualitativa para no comparar solo el cambio de checkpoint de
A-ESRGAN sin paso real por LaMa.

**A diferencia del daño sintético de 5.2/5.4**, esta máscara es una decisión subjetiva de una
persona y no reproducible bit a bit entre sesiones: dejarlo anotado así en el apartado 6.1 de la
memoria.


In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js

MAX_LADO_ANOTACION = 900   # tamaño de visualización del lienzo; no afecta a la resolución final

JS_PAINT_MASK = """
async function paintMask(imgDataUrl, dispW, dispH, brushInit) {
  const div = document.createElement('div');
  div.style.fontFamily = 'sans-serif';
  const instrucciones = document.createElement('p');
  instrucciones.innerText = 'Pinta sobre las zonas dañadas (roturas, arañazos, manchas). '
    + 'Arrastra el ratón (o el dedo) para pintar.';
  div.appendChild(instrucciones);

  const controls = document.createElement('div');
  controls.style.marginBottom = '6px';
  const brushLabel = document.createElement('label');
  brushLabel.innerText = 'Grosor: ';
  const brushInput = document.createElement('input');
  brushInput.type = 'range'; brushInput.min = 3; brushInput.max = 80; brushInput.value = brushInit;
  controls.appendChild(brushLabel); controls.appendChild(brushInput);

  const btnUndo = document.createElement('button'); btnUndo.innerText = 'Deshacer';
  btnUndo.style.marginLeft = '10px';
  const btnClear = document.createElement('button'); btnClear.innerText = 'Borrar todo';
  btnClear.style.marginLeft = '6px';
  const btnDone = document.createElement('button'); btnDone.innerText = 'Guardar máscara';
  btnDone.style.marginLeft = '16px'; btnDone.style.fontWeight = 'bold';
  controls.appendChild(btnUndo); controls.appendChild(btnClear); controls.appendChild(btnDone);
  div.appendChild(controls);

  const wrap = document.createElement('div');
  wrap.style.position = 'relative';
  wrap.style.width = dispW + 'px';
  wrap.style.height = dispH + 'px';
  wrap.style.border = '1px solid #999';

  const imgCanvas = document.createElement('canvas');
  imgCanvas.width = dispW; imgCanvas.height = dispH;
  imgCanvas.style.position = 'absolute'; imgCanvas.style.left = '0'; imgCanvas.style.top = '0';

  const overlayCanvas = document.createElement('canvas');
  overlayCanvas.width = dispW; overlayCanvas.height = dispH;
  overlayCanvas.style.position = 'absolute'; overlayCanvas.style.left = '0'; overlayCanvas.style.top = '0';
  overlayCanvas.style.cursor = 'crosshair';
  overlayCanvas.style.touchAction = 'none';

  wrap.appendChild(imgCanvas); wrap.appendChild(overlayCanvas);
  div.appendChild(wrap);
  document.body.appendChild(div);

  const maskCanvas = document.createElement('canvas');   // máscara binaria, no visible
  maskCanvas.width = dispW; maskCanvas.height = dispH;
  const maskCtx = maskCanvas.getContext('2d');
  maskCtx.fillStyle = 'black'; maskCtx.fillRect(0, 0, dispW, dispH);

  const octx = overlayCanvas.getContext('2d');
  const ictx = imgCanvas.getContext('2d');

  const img = new Image();
  await new Promise((res) => { img.onload = res; img.src = imgDataUrl; });
  ictx.drawImage(img, 0, 0, dispW, dispH);
  octx.drawImage(img, 0, 0, dispW, dispH);

  let pintando = false;
  let ultimo = null;
  let historial = [];

  function guardarHistorial() {
    historial.push(maskCtx.getImageData(0, 0, dispW, dispH));
    if (historial.length > 30) historial.shift();
  }

  function redibujarOverlay() {
    octx.clearRect(0, 0, dispW, dispH);
    octx.drawImage(img, 0, 0, dispW, dispH);
    octx.save();
    octx.globalAlpha = 0.45;
    octx.drawImage(maskCanvas, 0, 0);
    octx.restore();
  }

  function pos(e) {
    const r = overlayCanvas.getBoundingClientRect();
    const t = (e.touches && e.touches.length) ? e.touches[0] : e;
    return [t.clientX - r.left, t.clientY - r.top];
  }

  function trazar(x0, y0, x1, y1) {
    const b = parseInt(brushInput.value);
    maskCtx.strokeStyle = 'white'; maskCtx.fillStyle = 'white';
    maskCtx.lineWidth = b; maskCtx.lineCap = 'round'; maskCtx.lineJoin = 'round';
    maskCtx.beginPath(); maskCtx.moveTo(x0, y0); maskCtx.lineTo(x1, y1); maskCtx.stroke();
    maskCtx.beginPath(); maskCtx.arc(x1, y1, b / 2, 0, 2 * Math.PI); maskCtx.fill();
  }

  function onDown(e) {
    e.preventDefault(); guardarHistorial(); pintando = true;
    ultimo = pos(e); trazar(ultimo[0], ultimo[1], ultimo[0], ultimo[1]); redibujarOverlay();
  }
  function onMove(e) {
    if (!pintando) return;
    e.preventDefault();
    const p = pos(e); trazar(ultimo[0], ultimo[1], p[0], p[1]); ultimo = p; redibujarOverlay();
  }
  function onUp() { pintando = false; }

  overlayCanvas.addEventListener('mousedown', onDown);
  overlayCanvas.addEventListener('mousemove', onMove);
  window.addEventListener('mouseup', onUp);
  overlayCanvas.addEventListener('touchstart', onDown);
  overlayCanvas.addEventListener('touchmove', onMove);
  overlayCanvas.addEventListener('touchend', onUp);

  btnUndo.onclick = () => {
    if (historial.length === 0) return;
    maskCtx.putImageData(historial.pop(), 0, 0);
    redibujarOverlay();
  };
  btnClear.onclick = () => {
    guardarHistorial();
    maskCtx.fillStyle = 'black'; maskCtx.fillRect(0, 0, dispW, dispH);
    redibujarOverlay();
  };

  return new Promise((resolve) => {
    btnDone.onclick = () => {
      const dataUrl = maskCanvas.toDataURL('image/png');
      div.remove();
      resolve(dataUrl);
    };
  });
}
"""


def _imagen_a_dataurl(img_pil):
    buf = io.BytesIO()
    img_pil.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()


def _dataurl_a_array(data_url):
    _, b64 = data_url.split(',', 1)
    return np.array(Image.open(io.BytesIO(base64.b64decode(b64))).convert('L'))


def anotar_mascara(ruta_imagen, ruta_mascara_salida, brush_inicial=22, forzar=False):
    """Herramienta interactiva de pintado de máscaras (Colab).

    Muestra la imagen en un lienzo HTML, deja pintar las zonas dañadas con el ratón/dedo y
    guarda la máscara binaria (0/255) a la resolución de ruta_imagen. Si ya existe una máscara
    en ruta_mascara_salida y forzar=False, la devuelve sin pedir anotación de nuevo, así el
    notebook se puede reejecutar sin repintar las imágenes cada vez.
    """
    ruta_mascara_salida = Path(ruta_mascara_salida)
    if ruta_mascara_salida.exists() and not forzar:
        return np.array(Image.open(ruta_mascara_salida).convert('L'))

    img = Image.open(ruta_imagen).convert('RGB')
    w, h = img.size
    escala = min(1.0, MAX_LADO_ANOTACION / max(w, h))
    disp_w, disp_h = max(1, round(w * escala)), max(1, round(h * escala))
    img_disp = img.resize((disp_w, disp_h), Image.LANCZOS)

    display(Javascript(JS_PAINT_MASK))
    data_url = eval_js(
        f"paintMask('{_imagen_a_dataurl(img_disp)}', {disp_w}, {disp_h}, {brush_inicial})"
    )
    mask_disp = _dataurl_a_array(data_url)
    mask = cv2.resize(mask_disp, (w, h), interpolation=cv2.INTER_NEAREST)
    mask = np.where(mask > 127, 255, 0).astype(np.uint8)
    cv2.imwrite(str(ruta_mascara_salida), mask)
    return mask


print('Herramienta de anotación lista. Se invoca desde la celda siguiente, una imagen cada vez.')


In [ ]:
def mask_coverage(mask_u8):
    return float((mask_u8 > 127).mean())

coberturas = {}
for stem in df_conjunto.index:
    ruta_img  = DIR_ENTRADA / f'{stem}.png'
    ruta_mask = DIR_MASCARA / f'{stem}.png'
    if ruta_mask.exists():
        mask = np.array(Image.open(ruta_mask).convert('L'))
        print(f'  {stem}: máscara ya existente ({mask_coverage(mask):.2%} de cobertura) — se omite.')
    else:
        print(f'Anotando {stem} ...')
        mask = anotar_mascara(ruta_img, ruta_mask)
        print(f'  máscara guardada — cobertura {mask_coverage(mask):.2%}')
    coberturas[stem] = mask_coverage(mask)
    if coberturas[stem] == 0:
        print(f'  [AVISO] {stem}: máscara vacía. Si es intencionado (foto sin daño localizado '
              'visible) está bien; si no, borra el fichero y repinta.')

df_conjunto['cobertura'] = pd.Series(coberturas)
df_conjunto.to_csv(DIR_METRICAS / 'conjunto_6.csv')
n_vacias = int((df_conjunto['cobertura'] == 0).sum())
print(f'\n{len(df_conjunto)} máscaras listas en {DIR_MASCARA} ({n_vacias} vacías).')
print(f'Cobertura — media {df_conjunto["cobertura"].mean():.2%} | '
      f'min {df_conjunto["cobertura"].min():.2%} | max {df_conjunto["cobertura"].max():.2%}')


In [ ]:
# Revisión visual rápida de la anotación: imagen con la máscara superpuesta en rojo.
muestras_qa = df_conjunto.index[:4]
fig, axes = plt.subplots(1, len(muestras_qa), figsize=(3 * len(muestras_qa), 3.2))
if len(muestras_qa) == 1:
    axes = [axes]
for ax, stem in zip(axes, muestras_qa):
    img  = cv2.cvtColor(cv2.imread(str(DIR_ENTRADA / f'{stem}.png')), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(DIR_MASCARA / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    overlay = img.copy()
    overlay[mask > 127] = (0.55 * np.array([255, 0, 0]) + 0.45 * overlay[mask > 127]).astype(np.uint8)
    ax.imshow(overlay)
    ax.set_title(f'{stem[:16]} — {df_conjunto.loc[stem, "cobertura"]:.1%}', fontsize=7)
    ax.axis('off')
plt.suptitle('Revisión de la máscara pintada (rojo = daño marcado)', fontsize=9, fontweight='bold')
plt.tight_layout()
ruta_fig = DIR_FIGURAS / 'qa_mascaras_6.png'
fig.savefig(ruta_fig, dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)


## 4. Etapa 1 — inpainting con LaMa

Misma clase `EjecutorLaMa` que en 5.4: ambos modelos LaMa se ejecutan sobre la **misma** entrada
y la **misma** máscara pintada a mano en la sección 3. El ajustado carga el `state_dict` del
checkpoint de 5.2 sobre el módulo JIT de `SimpleLama`. La salida viene rellenada a múltiplo de 8;
se recorta al tamaño original en vez de reescalarla, para no introducir un remuestreo que no
forma parte del pipeline.


In [ ]:
from simple_lama_inpainting import SimpleLama

class EjecutorLaMa:
    """Envuelve SimpleLama para reutilizar el modelo entre imágenes.

    ckpt=None -> pesos big-lama preentrenados.
    ckpt=ruta -> state_dict del ajuste fino del apartado 5.2.
    """
    def __init__(self, ckpt=None):
        self.lama = SimpleLama()
        if ckpt is not None:
            estado = torch_load(ckpt, map_location='cpu')
            info = self.lama.model.load_state_dict(estado, strict=False)
            print(f'  checkpoint cargado ({Path(ckpt).name}) — '
                  f'{len(info.missing_keys)} claves ausentes, '
                  f'{len(info.unexpected_keys)} inesperadas')
        self.lama.model.to(DEVICE).eval()

    def __call__(self, img, mask):
        out = self.lama(img, mask).convert('RGB')
        if out.size != img.size:          # SimpleLama rellena a múltiplo de 8
            out = out.crop((0, 0, img.size[0], img.size[1]))
        return out

    def liberar(self):
        del self.lama
        gc.collect()
        torch.cuda.empty_cache()


def inpaintar_conjunto(ejecutor, dir_salida):
    dir_salida = Path(dir_salida)
    dir_salida.mkdir(parents=True, exist_ok=True)
    for stem in df_conjunto.index:
        img  = Image.open(DIR_ENTRADA / f'{stem}.png').convert('RGB')
        mask = Image.open(DIR_MASCARA / f'{stem}.png').convert('L')
        ejecutor(img, mask).save(dir_salida / f'{stem}.png')
    print(f'  {len(df_conjunto)} imágenes → {dir_salida}')

print('LaMa preentrenado:')
lama_pre = EjecutorLaMa(None)
inpaintar_conjunto(lama_pre, DIR_LAMA_PRE)
lama_pre.liberar(); del lama_pre

print('LaMa ajustado (5.2):')
assert PTH_LAMA_FT.exists(), f'No encuentro el checkpoint de LaMa ajustado en {PTH_LAMA_FT}'
lama_ft = EjecutorLaMa(PTH_LAMA_FT)
inpaintar_conjunto(lama_ft, DIR_LAMA_FT)
lama_ft.liberar(); del lama_ft

print('\nEtapa de inpainting completada para las dos configuraciones.')


## 5. Etapa 2 — super-resolución ×4 con A-ESRGAN

Se reutiliza sin cambios la función de inferencia por tiles de la Fase 4d/5.4, de modo que esta
etapa sea idéntica a la que se validó de forma aislada en 5.3. La entrada es la salida de LaMa de
cada configuración: el pipeline preentrenado consume la salida del LaMa preentrenado y el
ajustado la del LaMa ajustado.


In [ ]:
from basicsr.archs.rrdbnet_arch import RRDBNet

def cargar_modelo_sr(pth_path):
    """Carga un checkpoint RRDB en el dispositivo activo y lo pone en modo eval."""
    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                  num_block=23, num_grow_ch=32, scale=4)
    ckpt = torch_load(pth_path, map_location='cpu')
    estado = ckpt.get('params_ema', ckpt.get('params', ckpt))
    net.load_state_dict(estado, strict=True)
    return net.eval().to(DEVICE)


def _inferir_tiles(modelo, t, tile, tile_pad):
    """Inferencia por tiles con solapamiento para evitar OOM y artefactos de borde."""
    b, c, h, w = t.shape
    escala = 4
    out = torch.zeros(b, c, h * escala, w * escala, device=t.device)
    tiles_h = max(1, (h + tile - 1) // tile)
    tiles_w = max(1, (w + tile - 1) // tile)
    for i in range(tiles_h):
        for j in range(tiles_w):
            y0 = max(0, i * tile - tile_pad); y1 = min(h, (i + 1) * tile + tile_pad)
            x0 = max(0, j * tile - tile_pad); x1 = min(w, (j + 1) * tile + tile_pad)
            sr_patch = modelo(t[:, :, y0:y1, x0:x1])
            oy0 = (y0 + (tile_pad if i > 0 else 0)) * escala
            oy1 = (y1 - (tile_pad if i < tiles_h - 1 else 0)) * escala
            ox0 = (x0 + (tile_pad if j > 0 else 0)) * escala
            ox1 = (x1 - (tile_pad if j < tiles_w - 1 else 0)) * escala
            py0 = (tile_pad if i > 0 else 0) * escala; py1 = py0 + (oy1 - oy0)
            px0 = (tile_pad if j > 0 else 0) * escala; px1 = px0 + (ox1 - ox0)
            out[:, :, oy0:oy1, ox0:ox1] = sr_patch[:, :, py0:py1, px0:px1]
    return out


def superresolver(modelo, dir_lr, dir_salida, tile=512, tile_pad=32):
    dir_salida = Path(dir_salida); dir_salida.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        for stem in df_conjunto.index:
            img_bgr = cv2.imread(str(Path(dir_lr) / f'{stem}.png'), cv2.IMREAD_COLOR)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
            t = torch.from_numpy(img_rgb.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
            out = _inferir_tiles(modelo, t, tile, tile_pad)
            sr = (out.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
            cv2.imwrite(str(dir_salida / f'{stem}.png'), cv2.cvtColor(sr, cv2.COLOR_RGB2BGR))
    print(f'  {len(df_conjunto)} imágenes → {dir_salida}')


print('A-ESRGAN preentrenado sobre la salida del LaMa preentrenado:')
modelo_sr_pre = cargar_modelo_sr(PTH_AESRGAN_PRE)
superresolver(modelo_sr_pre, DIR_LAMA_PRE, DIR_SR_PRE)
del modelo_sr_pre; gc.collect(); torch.cuda.empty_cache()

print('A-ESRGAN brazo F_flick (iter 400) sobre la salida del LaMa ajustado:')
modelo_sr_ft = cargar_modelo_sr(PTH_AESRGAN_FT)
superresolver(modelo_sr_ft, DIR_LAMA_FT, DIR_SR_FT)
del modelo_sr_ft; gc.collect(); torch.cuda.empty_cache()

print('\nPipeline completo ejecutado en las dos configuraciones sobre fotografía histórica real.')


## 6. Protocolo de medida sin referencia — NIQE, BRISQUE, Ma, PI y MUSIQ

Sin imagen de alta resolución original, no hay PSNR, SSIM ni LPIPS. Se reportan cuatro métricas
ciegas, su combinación (PI) y una quinta de calidad perceptual profunda:

- **NIQE** (Mittal et al., 2013) — estadísticas de escena natural, sin entrenamiento con opiniones
  humanas. Implementación de `basicsr.metrics.niqe`, patrón oficial del script de BasicSR: sobre
  el array BGR que devuelve `cv2.imread`, sin normalizar.
- **BRISQUE** — misma familia que NIQE, pero "opinion-aware" (entrenada con puntuaciones humanas
  sobre la base LIVE). Si coincide con NIQE, refuerza la lectura; si diverge, es un dato en sí
  mismo. Vía `pyiqa`.
- **Ma (NRQM)** — métrica de Ma et al. (2017) para super-resolución, orientada a nitidez/textura
  percibida; a diferencia de las dos anteriores, **mayor es mejor**. Vía `pyiqa`.
- **PI (Perceptual Index)** = 0,5·((10−Ma)+NIQE) — la métrica ciega estándar de la literatura
  PIRM-SR/ESRGAN ya citada en el capítulo 2 (Wang et al., 2018; Wang et al., 2021). Se calcula a
  mano combinando **nuestro propio NIQE** (basicsr) con Ma (pyiqa), para no mezclar en la misma
  cifra el NIQE de basicsr con el NIQE interno de pyiqa, que usan parámetros distintos.
- **MUSIQ** (Ke et al., 2021) — transformer multi-escala entrenado sobre puntuaciones humanas
  de calidad (KonIQ-10k), a diferencia de NIQE/BRISQUE no es estadística de escena natural sin
  supervisar sino una predicción directa de opinión de calidad tipo MOS — **mayor es mejor**.
  Vía `pyiqa`. Se añade para contrastar con las cuatro anteriores: si un caso mejora a ojo pero
  NIQE/BRISQUE/Ma/PI lo penalizan (como `img24` en la sección 9.4), MUSIQ indica si una métrica
  más alineada con opinión humana coincide con esas cuatro o con la percepción visual.

Las cinco se calculan sobre las **tres condiciones** de la introducción — original degradada,
salida del pipeline preentrenado, salida del pipeline ajustado —, pareadas por imagen. Ninguna
está calibrada sobre fotografía histórica: triangulan el resultado, no lo hacen independiente del
dominio (ver advertencia de la introducción).


In [ ]:
from basicsr.metrics.niqe import calculate_niqe

def niqe_de(ruta_png):
    """NIQE de una imagen guardada en disco, siguiendo el patrón oficial de BasicSR
    (scripts/metrics/calculate_niqe.py): se lee con cv2.imread (BGR, uint8) y no se normaliza."""
    img = cv2.imread(str(ruta_png), cv2.IMREAD_COLOR)
    assert img is not None, f'No se pudo leer {ruta_png}'
    return float(calculate_niqe(img, crop_border=0, input_order='HWC', convert_to='y'))


CONDICIONES_NIQE = {
    'Original degradada':      DIR_ENTRADA,
    'Pipeline preentrenado':   DIR_SR_PRE,
    'Pipeline ajustado':       DIR_SR_FT,
}

registros_niqe = []
for stem in df_conjunto.index:
    for cond, directorio in CONDICIONES_NIQE.items():
        ruta = Path(directorio) / f'{stem}.png'
        registros_niqe.append({'imagen': stem, 'condicion': cond, 'NIQE': niqe_de(ruta)})
    print(f'  {stem}: NIQE calculado en las tres condiciones.')

df_niqe = pd.DataFrame(registros_niqe)
print(f'\n{len(df_niqe)} filas = {len(df_conjunto)} imágenes × {len(CONDICIONES_NIQE)} condiciones')
df_niqe.head(9)


In [ ]:
import pyiqa

try:
    metric_brisque = pyiqa.create_metric('brisque', device=DEVICE)
    metric_ma = pyiqa.create_metric('nrqm', device=DEVICE)
    metric_musiq = pyiqa.create_metric('musiq', device=DEVICE)
except Exception:
    print('No se pudo crear alguna métrica. Métricas disponibles en esta versión de pyiqa:')
    print(sorted(pyiqa.list_models()))
    raise

print('BRISQUE, Ma (nrqm) y MUSIQ listos en', DEVICE)


In [ ]:
# Limpieza defensiva: si esta celda ya falló antes en esta misma sesión, IPython puede seguir
# reteniendo el traceback de la excepción anterior (sys.last_traceback) — y con él, los tensores
# de GPU que causaron el OOM. empty_cache() no puede liberar memoria todavía referenciada por
# Python, así que hay que romper esa referencia explícitamente antes de reintentar.
import sys
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        setattr(sys, _attr, None)
gc.collect(); torch.cuda.empty_cache()
if torch.cuda.is_available():
    libre, total = torch.cuda.mem_get_info()
    print(f'GPU libre antes de empezar: {libre / 1024**3:.2f} / {total / 1024**3:.2f} GiB')
    if libre / total < 0.15:
        print('[AVISO] Queda muy poca memoria de GPU libre. Si esta celda vuelve a fallar por '
              'OOM, lo más fiable es Entorno de ejecución → Reiniciar sesión y volver a ejecutar '
              'desde la sección 6 (las salidas de LaMa/SR en Drive no se pierden).')

def _es_oom(err):
    return 'out of memory' in str(err).lower()

def _metricas_gpu(ruta):
    with torch.no_grad():
        b = float(metric_brisque(ruta).item())
        m = float(metric_ma(ruta).item())
        mq = float(metric_musiq(ruta).item())
    return b, m, mq

# Instancias de CPU independientes para el fallback: pyiqa guarda internamente el dispositivo con
# el que se creó la métrica (`create_metric(..., device=...)`) y lo usa para mover la imagen de
# entrada; llamar a `.to('cpu')` sobre el módulo NO cambia ese atributo interno, así que el
# cálculo seguía intentándose en GPU y volvía a agotar memoria. Se crean perezosamente (una sola
# vez) solo si de verdad hace falta el fallback.
_metricas_cpu_cache = {}

def _metricas_cpu(ruta):
    if not _metricas_cpu_cache:
        print('  [AVISO] Creando copias en CPU de BRISQUE/Ma/MUSIQ (una sola vez, más lento a partir de aquí)...')
        _metricas_cpu_cache['brisque'] = pyiqa.create_metric('brisque', device='cpu')
        _metricas_cpu_cache['ma'] = pyiqa.create_metric('nrqm', device='cpu')
        _metricas_cpu_cache['musiq'] = pyiqa.create_metric('musiq', device='cpu')
    with torch.no_grad():
        b = float(_metricas_cpu_cache['brisque'](ruta).item())
        m = float(_metricas_cpu_cache['ma'](ruta).item())
        mq = float(_metricas_cpu_cache['musiq'](ruta).item())
    return b, m, mq

registros_ciegas = []
n_fallback_cpu = 0
n_fallidas = 0
for stem in df_conjunto.index:
    for cond, directorio in CONDICIONES_NIQE.items():
        ruta = str(Path(directorio) / f'{stem}.png')
        brisque_val = ma_val = musiq_val = float('nan')
        try:
            brisque_val, ma_val, musiq_val = _metricas_gpu(ruta)
        except RuntimeError as e:
            if not _es_oom(e):
                raise
            e = None
            print(f'  [AVISO] CUDA sin memoria en {stem} ({cond}) — liberando caché y reintentando...')
            gc.collect(); torch.cuda.empty_cache()
            try:
                brisque_val, ma_val, musiq_val = _metricas_gpu(ruta)
            except RuntimeError as e2:
                if not _es_oom(e2):
                    raise
                e2 = None
                print(f'  [AVISO] Sigue sin memoria — calculando {stem} ({cond}) en CPU (más lento).')
                gc.collect(); torch.cuda.empty_cache()
                try:
                    brisque_val, ma_val, musiq_val = _metricas_cpu(ruta)
                    n_fallback_cpu += 1
                except Exception as e3:
                    print(f'  [AVISO] También falló en CPU para {stem} ({cond}): {e3!r} — se deja '
                          'como NaN, revisar a mano más adelante.')
                    e3 = None
                    n_fallidas += 1
        registros_ciegas.append({'imagen': stem, 'condicion': cond,
                                 'BRISQUE': brisque_val, 'Ma': ma_val, 'MUSIQ': musiq_val})
        # BRISQUE/Ma/MUSIQ (pyiqa) reservan memoria de forma que se fragmenta rápido en GPUs
        # pequeñas (T4 ~15 GiB) al encadenar muchas imágenes en el mismo proceso; liberar tras
        # cada imagen evita que un notebook con muchas fotos acabe agotando la memoria a mitad.
        gc.collect(); torch.cuda.empty_cache()
    print(f'  {stem}: BRISQUE, Ma y MUSIQ calculados en las tres condiciones.')

if n_fallback_cpu:
    print(f'\n[AVISO] {n_fallback_cpu} cálculo(s) de BRISQUE/Ma/MUSIQ se hicieron en CPU por falta '
          'de memoria de GPU — no afecta al valor, solo al tiempo de cómputo.')
if n_fallidas:
    print(f'\n[AVISO] {n_fallidas} cálculo(s) quedaron como NaN tras fallar también en CPU — '
          'revisar antes de dar los resultados por buenos.')

df_ciegas_extra = pd.DataFrame(registros_ciegas)
df_niqe = df_niqe.merge(df_ciegas_extra, on=['imagen', 'condicion'], how='left')
df_niqe['PI'] = 0.5 * ((10 - df_niqe['Ma']) + df_niqe['NIQE'])

df_niqe.to_csv(DIR_METRICAS / 'metricas_ciegas_por_imagen_6.csv', index=False)
print(f'\n{len(df_niqe)} filas con NIQE, BRISQUE, Ma, PI y MUSIQ.')
df_niqe.head(9)


## 7. Resultados

### 7.1 Media por condición


In [ ]:
ORDEN_COND = list(CONDICIONES_NIQE.keys())
METRICAS_CIEGAS = {'NIQE': '↓', 'BRISQUE': '↓', 'Ma': '↑', 'PI': '↓', 'MUSIQ': '↑'}

resumen_ciegas = {}
for met in METRICAS_CIEGAS:
    tabla = (df_niqe.groupby('condicion')[met].agg(['mean', 'std', 'median']).reindex(ORDEN_COND))
    tabla.columns = [f'{met} medio', 'desv. típica', 'mediana']
    resumen_ciegas[met] = tabla
    print(f'{met} {METRICAS_CIEGAS[met]} por condición (n = {df_niqe["imagen"].nunique()})\n')
    print(tabla.to_string(float_format='{:.4f}'.format))
    print()
    tabla.to_csv(DIR_METRICAS / f'resumen_{met.lower()}_6.csv')


### 7.2 Contraste estadístico (Wilcoxon pareado)

Wilcoxon de rangos con signo sobre los pares por imagen, con **corrección de Bonferroni** — cada
métrica es su propia familia de tres comparaciones (original-preentrenado, original-ajustado,
preentrenado-ajustado): umbral corregido α = 0,05/3 ≈ 0,0167. El contraste con carga argumental
para la memoria (6.2/6.3) es siempre **preentrenado frente a ajustado**: es el único, en las
cinco métricas, en el que ambas condiciones están a la misma escala (ver advertencia de la
introducción).


In [ ]:
COMPARACIONES = [
    ('Original degradada',    'Pipeline preentrenado'),
    ('Original degradada',    'Pipeline ajustado'),
    ('Pipeline preentrenado', 'Pipeline ajustado'),
]
ALFA = 0.05
M_COMPARACIONES = len(COMPARACIONES)

filas_wx_ciegas = []
for met, direccion in METRICAS_CIEGAS.items():
    piv_met = df_niqe.pivot(index='imagen', columns='condicion', values=met)
    for a, b in COMPARACIONES:
        va, vb = piv_met[a].values, piv_met[b].values
        dif = vb - va
        mejora_b = (dif < 0) if direccion == '↓' else (dif > 0)
        if np.allclose(dif, 0):
            stat, p = np.nan, 1.0
        else:
            stat, p = wilcoxon(vb, va, alternative='two-sided')
        p_adj = min(1.0, p * M_COMPARACIONES)
        filas_wx_ciegas.append({
            'Métrica': met, 'A': a, 'B': b, 'n': len(piv_met),
            'Δ medio (B − A)': float(dif.mean()), 'Δ mediana (B − A)': float(np.median(dif)),
            'B mejora': f'{int(mejora_b.sum())}/{len(piv_met)}',
            'W': float(stat) if stat == stat else np.nan,
            'p': float(p), 'p Bonferroni': float(p_adj),
            'Significativo (α=0,05)': 'sí' if p_adj < ALFA else 'no',
        })

df_wx_ciegas = pd.DataFrame(filas_wx_ciegas)
print(f'Wilcoxon pareado, familia de {M_COMPARACIONES} comparaciones por métrica '
      f'(umbral corregido α = {ALFA / M_COMPARACIONES:.4f})\n')
print(df_wx_ciegas.to_string(index=False, float_format='{:.4f}'.format))
df_wx_ciegas.to_csv(DIR_METRICAS / 'wilcoxon_metricas_ciegas_6.csv', index=False)

fila_niqe_principal = df_wx_ciegas[(df_wx_ciegas['Métrica'] == 'NIQE') &
                                   (df_wx_ciegas['A'] == 'Pipeline preentrenado') &
                                   (df_wx_ciegas['B'] == 'Pipeline ajustado')].iloc[0]
p_contraste_principal = float(fila_niqe_principal['p Bonferroni'])
print(f'\nContraste principal NIQE (preentrenado vs. ajustado): p Bonferroni = {p_contraste_principal:.4f}')


## 8. Análisis cualitativo sobre rostros

El apartado 6.2 pide un análisis cualitativo de acompañamiento "incluido el comportamiento sobre
rostros, que conecta con 6.4" (consideraciones éticas). Se reutiliza el mismo detector que el
demostrador de 5.5 — el clasificador en cascada de Haar de OpenCV — y el mismo umbral: se marca
una imagen como «máscara solapa con rostro» cuando la máscara pintada cubre más del 2 % del
rectángulo de algún rostro detectado. No mide si la reconstrucción facial es correcta — no hay con
qué compararla — solo cuántas fotos tienen daño marcado sobre una región facial.


In [ ]:
UMBRAL_SOLAPE_ROSTRO = 0.02   # mismo criterio que el demostrador (5.5)

_detector_rostros = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def detectar_rostros(img_rgb):
    gris = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    return _detector_rostros.detectMultiScale(gris, scaleFactor=1.1, minNeighbors=5,
                                               minSize=(30, 30))

def solapa_con_rostro(mask_bool, rostros, umbral=UMBRAL_SOLAPE_ROSTRO):
    for (x, y, w, h) in rostros:
        area_rostro = mask_bool[y:y + h, x:x + w]
        if area_rostro.size and (area_rostro.mean() > umbral):
            return True
    return False


registros_rostros = []
for stem in df_conjunto.index:
    img_rgb = cv2.cvtColor(cv2.imread(str(DIR_ENTRADA / f'{stem}.png')), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(DIR_MASCARA / f'{stem}.png'), cv2.IMREAD_GRAYSCALE) > 127
    rostros = detectar_rostros(img_rgb)
    registros_rostros.append({
        'imagen': stem,
        'rostros_detectados': len(rostros),
        'mascara_solapa_rostro': solapa_con_rostro(mask, rostros) if len(rostros) else False,
    })

df_rostros = pd.DataFrame(registros_rostros).set_index('imagen')
df_rostros.to_csv(DIR_METRICAS / 'rostros_6.csv')

n_con_rostro = int((df_rostros['rostros_detectados'] > 0).sum())
n_solapa = int(df_rostros['mascara_solapa_rostro'].sum())
print(f'Rostros detectados en {n_con_rostro}/{len(df_rostros)} fotografías.')
print(f'De ellas, {n_solapa} tienen la máscara de daño solapando con un rostro detectado '
      f'(umbral {UMBRAL_SOLAPE_ROSTRO:.0%} del rectángulo facial).')
print('\nRecordatorio: el detector de Haar es clásico y puede fallar en retratos históricos con '
      'iluminación o pose difíciles; tratar estas cifras como orientativas, no exhaustivas, en '
      'el apartado 6.2/6.4.')


## 9. Figuras

### 9.1 Distribución de las métricas ciegas en las tres condiciones


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
colores = [OI_GREY, OI_ORANGE, OI_BLUE]
metricas_fig = ['NIQE', 'BRISQUE', 'Ma', 'PI', 'MUSIQ']
ejes = axes.ravel()

for ax, met in zip(ejes, metricas_fig):
    direccion = METRICAS_CIEGAS[met]
    piv_met = df_niqe.pivot(index='imagen', columns='condicion', values=met)
    datos = [piv_met[c].values for c in ORDEN_COND]
    bp = ax.boxplot(datos, patch_artist=True, widths=0.45,
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
                    flierprops=dict(marker='o', markersize=3, alpha=0.5))
    for patch, c in zip(bp['boxes'], colores):
        patch.set_facecolor(c); patch.set_alpha(0.75)

    rng = np.random.default_rng(42)
    for k, (vals, c) in enumerate(zip(datos, colores), start=1):
        ax.scatter(np.full(len(vals), k) + rng.uniform(-0.12, 0.12, size=len(vals)), vals,
                   color=c, edgecolors='white', linewidths=0.4, s=22, alpha=0.85, zorder=3)
    for stem in piv_met.index:
        ax.plot([1, 2, 3], piv_met.loc[stem, ORDEN_COND].values,
                color='#aaaaaa', linewidth=0.5, alpha=0.35, zorder=2)

    fila_p = df_wx_ciegas[(df_wx_ciegas['Métrica'] == met) &
                          (df_wx_ciegas['A'] == 'Pipeline preentrenado') &
                          (df_wx_ciegas['B'] == 'Pipeline ajustado')].iloc[0]
    p_adj_met = float(fila_p['p Bonferroni'])
    y0, y1 = min(v.min() for v in datos), max(v.max() for v in datos)
    step = max((y1 - y0) * 0.08, 1e-3)
    y_ann = y1 + step
    ax.plot([2, 2, 3, 3], [y_ann, y_ann + step * 0.4, y_ann + step * 0.4, y_ann],
            color='black', linewidth=1)
    ax.text(2.5, y_ann + step * 0.55,
            f'p = {p_adj_met:.3f}' + (' *' if p_adj_met < ALFA else ' n.s.'),
            ha='center', va='bottom', fontsize=8)
    ax.set_ylim(top=y_ann + step * 1.3)

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['Original\ndegradada', 'Pipeline\npreentrenado', 'Pipeline\najustado'],
                       fontsize=7)
    ax.set_ylabel(f'{met} {direccion}')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

for ax in ejes[len(metricas_fig):]:
    ax.axis('off')

fig.suptitle(f'Métricas ciegas sobre fotografía histórica real (n = {df_niqe["imagen"].nunique()}, '
             'p con corrección de Bonferroni por métrica)', fontsize=9, fontweight='bold')
plt.tight_layout()
ruta_fig = DIR_FIGURAS / 'metricas_ciegas_boxplot_6.png'
fig.savefig(ruta_fig, dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)


### 9.2 Comparación cualitativa

Cuatro fotografías: las dos con mayor mejora de NIQE del pipeline ajustado frente al preentrenado
y las dos con diferencia mínima, igual que en 5.4 — pero **restringido a imágenes con máscara no
vacía**. Sin esa restricción, la selección de "diferencia mínima" tiende a caer sobre fotos sin
daño marcado, donde LaMa no tuvo nada que reconstruir: la comparación acaba siendo solo entre
checkpoints de A-ESRGAN, no entre pipelines completos, que es lo que este apartado quiere mostrar.


In [ ]:
def _leer_rgb(ruta):
    img = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
    assert img is not None, f'No se pudo leer {ruta}'
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

piv_niqe = df_niqe.pivot(index='imagen', columns='condicion', values='NIQE')
delta_niqe = (piv_niqe['Pipeline preentrenado'] - piv_niqe['Pipeline ajustado']).dropna()

candidatos = df_conjunto.index[df_conjunto['cobertura'] > 0]
delta_candidatos = delta_niqe.loc[delta_niqe.index.intersection(candidatos)]
if len(delta_candidatos) < 4:
    print('[AVISO] Menos de 4 imágenes con máscara no vacía; se usan todas las disponibles.')
    delta_candidatos = delta_niqe

mejores = delta_candidatos.nlargest(2).index.tolist()
neutros = delta_candidatos.drop(index=mejores, errors='ignore').abs().nsmallest(2).index.tolist()
ejemplos = mejores + neutros
print(f'Ejemplos seleccionados de entre {len(delta_candidatos)} imágenes con máscara no vacía '
      f'(2 de mayor mejora + 2 de diferencia mínima): {ejemplos}')

cabeceras = ['Original degradada', 'Máscara pintada', 'Pipeline preentrenado', 'Pipeline ajustado']
fig, axes = plt.subplots(len(ejemplos), 4, figsize=(11, 2.4 * len(ejemplos)))
if len(ejemplos) == 1:
    axes = axes[np.newaxis, :]
for col, cab in enumerate(cabeceras):
    axes[0, col].set_title(cab, fontsize=8, fontweight='bold', pad=3)

for fila, stem in enumerate(ejemplos):
    original = _leer_rgb(DIR_ENTRADA / f'{stem}.png')
    mask     = cv2.imread(str(DIR_MASCARA / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    overlay  = original.copy()
    overlay[mask > 127] = (0.55 * np.array([255, 0, 0]) + 0.45 * overlay[mask > 127]).astype(np.uint8)
    sr_pre = _leer_rgb(DIR_SR_PRE / f'{stem}.png')
    sr_ft  = _leer_rgb(DIR_SR_FT / f'{stem}.png')
    for col, img in enumerate([original, overlay, sr_pre, sr_ft]):
        axes[fila, col].imshow(img)
        axes[fila, col].set_xticks([]); axes[fila, col].set_yticks([])
        for sp in axes[fila, col].spines.values():
            sp.set_visible(False)
    for col, cond in [(2, 'Pipeline preentrenado'), (3, 'Pipeline ajustado')]:
        axes[fila, col].text(0.02, 0.03, f'NIQE={piv_niqe.loc[stem, cond]:.2f}',
                             transform=axes[fila, col].transAxes, fontsize=7, color='white',
                             bbox=dict(facecolor='black', alpha=0.55, pad=1, edgecolor='none'))
    axes[fila, 0].text(0.02, 0.03, stem[:18], transform=axes[fila, 0].transAxes,
                       fontsize=7, color='white',
                       bbox=dict(facecolor='black', alpha=0.55, pad=1, edgecolor='none'))

plt.suptitle('Comparación cualitativa del pipeline completo — fotografía histórica real',
             fontsize=9, fontweight='bold', y=0.995)
plt.subplots_adjust(wspace=0.02, hspace=0.05, top=0.95, bottom=0.01, left=0.01, right=0.99)
ruta_fig = DIR_FIGURAS / 'cualitativa_6.png'
fig.savefig(ruta_fig, dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)


### 9.3 Detalle de la región reconstruida

Mismo criterio que en 5.4: recorte centrado en la zona de mayor densidad de daño, a resolución de
salida (×4), sobre las mismas cuatro fotografías de 9.2.


In [ ]:
FRAC_VENTANA = 0.40
LADO_MIN_DET = 96

def _alinear(img, tam_wh, interp=cv2.INTER_LANCZOS4):
    w, h = tam_wh
    if img.shape[1] == w and img.shape[0] == h:
        return img
    return cv2.resize(img, (w, h), interpolation=interp)

def ventana_danio(mask_lr, tam_wh, frac=FRAC_VENTANA, lado_min=LADO_MIN_DET):
    """Devuelve (y0, y1, x0, x1) de la ventana cuadrada con mayor densidad de daño."""
    w, h = tam_wh
    msk = (_alinear(mask_lr, (w, h), interp=cv2.INTER_NEAREST) > 127).astype(np.float32)
    lado = int(min(max(lado_min, round(min(w, h) * frac)), w, h))
    centrada = ((h - lado) // 2, (h - lado) // 2 + lado, (w - lado) // 2, (w - lado) // 2 + lado)
    if not msk.any():
        return centrada
    densidad = cv2.boxFilter(msk, -1, (lado, lado), normalize=True,
                             borderType=cv2.BORDER_CONSTANT)
    mitad = lado // 2
    interior = densidad[mitad:h - (lado - mitad) + 1, mitad:w - (lado - mitad) + 1]
    if interior.size == 0:
        return centrada
    iy, ix = np.unravel_index(int(np.argmax(interior)), interior.shape)
    return iy, iy + lado, ix, ix + lado


cabeceras = ['Entrada (recorte)', 'Pipeline preentrenado', 'Pipeline ajustado']
fig, axes = plt.subplots(len(ejemplos), 3, figsize=(8, 2.6 * len(ejemplos)))
if len(ejemplos) == 1:
    axes = axes[np.newaxis, :]
for col, cab in enumerate(cabeceras):
    axes[0, col].set_title(cab, fontsize=8, fontweight='bold', pad=3)

for fila, stem in enumerate(ejemplos):
    mask_lr = cv2.imread(str(DIR_MASCARA / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    sr_pre  = _leer_rgb(DIR_SR_PRE / f'{stem}.png')
    sr_ft   = _leer_rgb(DIR_SR_FT / f'{stem}.png')
    ent     = _alinear(_leer_rgb(DIR_ENTRADA / f'{stem}.png'),
                       (sr_pre.shape[1], sr_pre.shape[0]), interp=cv2.INTER_NEAREST)
    tam = (sr_pre.shape[1], sr_pre.shape[0])

    y0, y1, x0, x1 = ventana_danio(mask_lr, tam)
    _msk_sr = _alinear(mask_lr, tam, interp=cv2.INTER_NEAREST) > 127
    print(f'{stem}: ventana {x1 - x0}×{y1 - y0} px en ({x0}, {y0}) — '
          f'daño dentro {_msk_sr[y0:y1, x0:x1].mean():.1%} '
          f'frente a {_msk_sr.mean():.1%} en la imagen completa')

    for col, img in enumerate([ent, sr_pre, sr_ft]):
        axes[fila, col].imshow(img[y0:y1, x0:x1])
        axes[fila, col].set_xticks([]); axes[fila, col].set_yticks([])
        for sp in axes[fila, col].spines.values():
            sp.set_visible(False)

plt.subplots_adjust(wspace=0.02, hspace=0.05, top=0.95, bottom=0.01, left=0.01, right=0.99)
ruta_fig = DIR_FIGURAS / 'detalle_mascara_6.png'
fig.savefig(ruta_fig, dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)


### 9.4 Casos de mayor cobertura de máscara

Los ejemplos de 9.2/9.3 se eligen por diferencia de NIQE, no por cantidad de daño. Aquí se
muestran aparte las **5 fotografías históricas con mayor porcentaje de máscara pintada**
(`df_conjunto['cobertura']`), los casos más exigentes para el pipeline dentro del conjunto de 21 —
útil para ilustrar el comportamiento en el extremo superior de daño, no solo en los casos de
mejora típica.


In [ ]:
stems_max_cobertura = df_conjunto['cobertura'].nlargest(5).index.tolist()
print('Imágenes con mayor cobertura de máscara:')
for s in stems_max_cobertura:
    print(f'  {s}: {df_conjunto.loc[s, "cobertura"]:.2%}')

cabeceras = ['Original degradada', 'Máscara pintada', 'Pipeline preentrenado', 'Pipeline ajustado']
fig, axes = plt.subplots(len(stems_max_cobertura), 4, figsize=(11, 2.6 * len(stems_max_cobertura)))
if len(stems_max_cobertura) == 1:
    axes = axes[np.newaxis, :]
for col, cab in enumerate(cabeceras):
    axes[0, col].set_title(cab, fontsize=8, fontweight='bold', pad=3)

for fila, stem in enumerate(stems_max_cobertura):
    original = _leer_rgb(DIR_ENTRADA / f'{stem}.png')
    mask     = cv2.imread(str(DIR_MASCARA / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
    overlay  = original.copy()
    overlay[mask > 127] = (0.55 * np.array([255, 0, 0]) + 0.45 * overlay[mask > 127]).astype(np.uint8)
    sr_pre = _leer_rgb(DIR_SR_PRE / f'{stem}.png')
    sr_ft  = _leer_rgb(DIR_SR_FT / f'{stem}.png')
    for col, img in enumerate([original, overlay, sr_pre, sr_ft]):
        axes[fila, col].imshow(img)
        axes[fila, col].set_xticks([]); axes[fila, col].set_yticks([])
        for sp in axes[fila, col].spines.values():
            sp.set_visible(False)
    cobertura = df_conjunto.loc[stem, 'cobertura']
    axes[fila, 0].text(0.02, 0.03, f'{stem[:16]} ({cobertura:.1%})',
                       transform=axes[fila, 0].transAxes, fontsize=7, color='white',
                       bbox=dict(facecolor='black', alpha=0.55, pad=1, edgecolor='none'))
    for col, cond in [(2, 'Pipeline preentrenado'), (3, 'Pipeline ajustado')]:
        axes[fila, col].text(0.02, 0.03, f'NIQE={piv_niqe.loc[stem, cond]:.2f}',
                             transform=axes[fila, col].transAxes, fontsize=7, color='white',
                             bbox=dict(facecolor='black', alpha=0.55, pad=1, edgecolor='none'))

plt.suptitle('Casos de mayor daño anotado (5 mayor cobertura de máscara)',
             fontsize=9, fontweight='bold', y=0.995)
plt.subplots_adjust(wspace=0.02, hspace=0.05, top=0.96, bottom=0.01, left=0.01, right=0.99)
ruta_fig = DIR_FIGURAS / 'mayor_cobertura_6.png'
fig.savefig(ruta_fig, dpi=200, bbox_inches='tight')
plt.show()
print('Figura guardada:', ruta_fig)

tabla_max = (df_niqe[df_niqe['imagen'].isin(stems_max_cobertura)]
             .set_index(['imagen', 'condicion'])[['NIQE', 'BRISQUE', 'Ma', 'PI', 'MUSIQ']]
             .reindex(pd.MultiIndex.from_product([stems_max_cobertura, ORDEN_COND],
                                                  names=['imagen', 'condicion'])))
print('\nMétricas ciegas:\n')
print(tabla_max.to_string(float_format='{:.4f}'.format))


## 10. Exportación de resultados y síntesis

Un único `resultados_6.json` con todo lo necesario para redactar el apartado 6: el conjunto de
evaluación (incluida la trazabilidad del filtro histórico-vs-moderno), las medias y el contraste
de Wilcoxon de las cinco métricas ciegas, y el resumen de rostros.


In [ ]:
resultados = {
    'conjunto': {
        'n_total_carpeta': len(rutas_todas),
        'n_historicas': len(df_conjunto),
        'n_descartadas_degradacion_moderna': len(descartadas),
        'terminaciones_historicas': TERMINACIONES_HISTORICAS,
        'terminaciones_no_encontradas': no_encontradas,
        'n_mascara_vacia': int((df_conjunto['cobertura'] == 0).sum()),
        'imagenes_mascara_vacia': df_conjunto.index[df_conjunto['cobertura'] == 0].tolist(),
        'cobertura_media': float(df_conjunto['cobertura'].mean()),
        'nombres_calibracion_comprobados': sorted(NOMBRES_CALIBRACION),
        'solapamiento_con_calibracion': solapan,
    },
    'metricas_ciegas': {
        met: {
            'direccion': METRICAS_CIEGAS[met],
            'por_condicion': resumen_ciegas[met].round(4).to_dict(orient='index'),
        }
        for met in METRICAS_CIEGAS
    },
    'wilcoxon_metricas_ciegas': df_wx_ciegas.round(4).to_dict(orient='records'),
    'contraste_principal_niqe_preentrenado_vs_ajustado': {
        'p_bonferroni': p_contraste_principal,
        'significativo': p_contraste_principal < ALFA,
    },
    'rostros': {
        'n_con_rostro_detectado': n_con_rostro,
        'n_mascara_solapa_rostro': n_solapa,
        'umbral_solape': UMBRAL_SOLAPE_ROSTRO,
    },
    'ejemplos_cualitativos': {
        'seleccionados': ejemplos,
        'mejor_mejora_niqe': mejores,
        'diferencia_minima_niqe': neutros,
        'n_candidatos_mascara_no_vacia': len(delta_candidatos),
    },
}

with open(DIR_METRICAS / 'resultados_6.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print('=' * 78)
print('SÍNTESIS — Capítulo 6: evaluación con fotografía histórica real')
print('=' * 78)
print(f'\nConjunto: {len(df_conjunto)} fotografías verdaderamente históricas de las '
      f'{len(rutas_todas)} encontradas en {DIR_REAL.name} '
      f'({len(descartadas)} descartadas por ser degradación moderna artificial).')
print(f'{resultados["conjunto"]["n_mascara_vacia"]}/{len(df_conjunto)} tienen máscara de daño '
      'vacía (sin daño localizado visible) — excluidas de la selección de ejemplos cualitativos.')

print('\nMétricas ciegas (NIQE, BRISQUE, Ma, PI, MUSIQ) por condición:')
for met, direccion in METRICAS_CIEGAS.items():
    tabla = resumen_ciegas[met]
    print(f'  {met} {direccion}: ' + ' | '.join(
        f'{cond}={tabla.loc[cond, f"{met} medio"]:.3f}' for cond in ORDEN_COND))

print(f'\nContraste limpio (preentrenado vs. ajustado, mismo par LaMa+SR salvo el checkpoint '
      f'de A-ESRGAN), p con corrección de Bonferroni (familia de {M_COMPARACIONES}):')
for met in METRICAS_CIEGAS:
    fila = df_wx_ciegas[(df_wx_ciegas['Métrica'] == met) &
                        (df_wx_ciegas['A'] == 'Pipeline preentrenado') &
                        (df_wx_ciegas['B'] == 'Pipeline ajustado')].iloc[0]
    print(f'  {met}: p = {fila["p Bonferroni"]:.4f} '
          f'({"significativo" if fila["p Bonferroni"] < ALFA else "no significativo"} a α=0,05)')

print(f'\nRostros: {n_con_rostro}/{len(df_rostros)} fotografías con al menos un rostro detectado; '
      f'{n_solapa} con la máscara de daño solapando un rostro (umbral '
      f'{UMBRAL_SOLAPE_ROSTRO:.0%}).')

print('\nAdvertencias a tener presentes en la redacción del apartado 6:')
print('  · Las cinco métricas ciegas (NIQE, BRISQUE, Ma/NRQM, PI derivado de ellas y MUSIQ)')
print('    están entrenadas o calibradas sobre fotografía natural moderna, no histórica — un')
print('    sesgo de dominio compartido por las cinco, no solo por NIQE. Una mejora numérica no')
print('    implica')
print('    necesariamente una mejora perceptual sobre este dominio (ver ejemplo de textura tipo')
print('    grano de película en zona de velo/reflejo, apartado 6.3).')
if not NOMBRES_CALIBRACION:
    print('  · La separación frente al corpus de calibración espectral de 5.3 NO se ha podido')
    print('    comprobar automáticamente (NOMBRES_CALIBRACION vacío) — confirmar a mano antes')
    print('    del depósito.')
elif solapan:
    print(f'  · ¡ATENCIÓN! {len(solapan)} imágenes solapan con el corpus de calibración: {solapan}.')
print(f'  · {resultados["conjunto"]["n_mascara_vacia"]} fotografías tienen máscara vacía — se')
print('    excluyeron del pool de selección de ejemplos cualitativos (sección 9.2) para no medir')
print('    solo diferencias de checkpoint de A-ESRGAN.')
print('  · Las máscaras son anotación manual (sección 3), no reproducibles bit a bit entre')
print('    ejecuciones distintas; si ya existen en Drive, esta celda las reutiliza sin repintar.')
print('\nResultados completos exportados a', DIR_METRICAS / 'resultados_6.json')


## 11. Notas de reproducibilidad

- **Origen del conjunto.** `datasets/Vintage_Degraded` contiene 54 fotografías con el mismo
  patrón de nombre (`low-resolution-photographs_imgN`), mezclando degradación histórica real y
  degradación moderna artificial. La sección 2 filtra a las 21 verdaderamente históricas **antes**
  de anotar o procesar nada, usando `TERMINACIONES_HISTORICAS` (lista de terminaciones de nombre
  de fichero identificadas a mano por Javier). Las 33 fotografías moderas nunca se anotan, ni se
  pasan por LaMa/SR, ni entran en ninguna tabla o figura de este notebook.
- **Máscaras.** Anotación manual vía `anotar_mascara` (herramienta HTML5 canvas embebida con
  `google.colab.output.eval_js`, solo funciona en Colab). No son reproducibles bit a bit entre
  ejecuciones distintas de la misma persona, y mucho menos entre dos personas distintas. La celda
  de anotación es resumible: si el fichero de máscara ya existe en `DIR_MASCARA`, no se repinta.
  4 de las 21 fotografías históricas (`img110`, `img173`, `img18`, `img23` — comprobar contra
  `resultados_6.json → conjunto → imagenes_mascara_vacia` si la lista cambia) quedaron con máscara
  vacía tras la anotación; es intencionado (sin daño localizado claramente delimitable), no un
  error, pero se excluyen del pool de ejemplos cualitativos de la sección 9.2.
- **LaMa y SR no son resumibles.** A diferencia de la anotación de máscaras, las celdas de
  inpainting y super-resolución reprocesan las 21 imágenes en cada ejecución del notebook (no
  comprueban si ya existe una salida en `DIR_LAMA_*` / `DIR_SR_*`). Si solo se quiere repetir el
  análisis de métricas sobre salidas ya generadas, basta con no reejecutar esas dos secciones.
- **Métricas ciegas y dependencia de `pyiqa`.** NIQE se calcula con la implementación de
  `basicsr` (igual que en 5.4/5.5, para no mezclar dos implementaciones de NIQE en una misma
  figura); BRISQUE, Ma/NRQM y MUSIQ se calculan con `pyiqa.create_metric(...)`, y PI se deriva
  de NIQE y Ma como `0.5 · ((10 − Ma) + NIQE)`, replicando la definición estándar del PI del
  reto PIRM 2018 pero con la implementación propia de NIQE. Las cuatro métricas de partida
  (NIQE, BRISQUE, Ma, MUSIQ) están entrenadas o calibradas sobre fotografía natural moderna —
  ninguna de las cuatro, ni por tanto PI, es un juicio de calidad neutral sobre fotografía
  histórica; tratar los resultados numéricos como un indicio, no como una medida absoluta de
  restauración. MUSIQ, a diferencia de NIQE/BRISQUE, está entrenada para predecir opinión
  humana (MOS) directamente — más cercana a percepción, pero sobre el mismo tipo de fotografía
  moderna, así que el sesgo de dominio no desaparece.
- **Corpus de calibración espectral (5.3).** `NOMBRES_CALIBRACION` en la sección 2 debe rellenarse
  a mano con los nombres de fichero del corpus usado para calibrar `psd_objetivo.npz` en la fase
  5.3, para poder comprobar automáticamente que ninguna de las 21 fotografías de evaluación se
  usó también para calibrar el módulo de degradación. Si se deja vacío, la celda avisa pero no
  puede garantizar la separación — confirmarlo a mano antes del depósito.
- **Detección de rostros.** Haar cascade clásico (`haarcascade_frontalface_default.xml`), mismo
  criterio que el demostrador de 5.5. Puede fallar en retratos con pose o iluminación difíciles,
  frecuentes en fotografía histórica; las cifras de la sección 8 son orientativas.
- **Reejecución completa.** Primera celda de código (entorno + dependencias + descargas) → sección 2 (filtro histórico,
  ~segundos) → sección 3 (anotación, solo pide interacción si faltan máscaras) → secciones 4–5
  (LaMa + SR, minutos en GPU T4) → secciones 6–9 (métricas y figuras) → sección 10 (exportación).
  Todas las rutas de salida cuelgan de `ROOT/_out/07/` (efímeras en Colab; usar
  `montar_drive_opcional()` para persistirlas), así que una reejecución completa sobrescribe
  los CSV y PNG anteriores salvo las máscaras ya pintadas.
